# Knowledgebase creation Notebook
---


## 1. Setup & Configuration


In [ ]:
import os
import re
import uuid
import warnings
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from pymupdf import open as pdf_open

In [ ]:
warnings.filterwarnings("ignore")

# -- Load environment --
load_dotenv(dotenv_path=r"d:\coding\new\.env", override=True)

api_key = os.getenv("OPENAI_API_KEY")
qdrant_url = os.getenv("QDRANT_HOST")
qdrant_api_key = os.getenv("QDRANT_API_KEY")

if not api_key or "YOUR_" in api_key:
    raise ValueError("Invalid or missing OPENAI_API_KEY in .env")

# -- Initialize clients --
openai_client = OpenAI(api_key=api_key)
qdrant_client = QdrantClient(
    url=f"https://{qdrant_url}:6333",
    api_key=qdrant_api_key,
    timeout=120
)

## 2. Constants & Helper Functions


In [ ]:
# ── Constants ──
PDF_FOLDER = r".\Acts"
EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 3072

# Collection names (one per strategy)
COLLECTIONS = {
    "semantic":  "legal_semantic",
    "fixed":     "legal_fixed",
    "recursive": "legal_recursive",
}

# ── Tokenizer ──
encoding = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    """Count tokens using the cl100k_base tokenizer."""
    return len(encoding.encode(text))

# ── Embedding function (same as rag.ipynb) ──
def embed_texts(texts):
    """Embed a list of texts using OpenAI text-embedding-3-large."""
    response = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in response.data]


## 3. Chunking Strategies


---
### 3a. Semantic Chunking

In [ ]:
def semantic_chunk_pdfs(pdf_folder):
    """
    SEMANTIC CHUNKING — preserves legal section boundaries.
    
    Strategy:
    1. Extract full text from each PDF
    2. Split by section number patterns (e.g. '104. ...')
    3. Within each section, group paragraphs up to max_tokens
    
    Returns: (documents, metadatas, ids)
    """
    documents, metadatas, ids = [], [], []
    
    def split_by_sections(text):
        """Split legal document by section numbers."""
        pattern = r"\n(?=\d{1,4}\.\s)"
        sections = re.split(pattern, text)
        return [sec.strip() for sec in sections if sec.strip()]
    
    def semantic_chunk_section(section_text, max_tokens=800):
        """Further split large sections into semantically grouped chunks.
        Keeps paragraphs intact within token limits."""
        paragraphs = section_text.split("\n")
        chunks, current_chunk = [], ""
        for para in paragraphs:
            candidate = current_chunk + "\n" + para if current_chunk else para
            if count_tokens(candidate) < max_tokens:
                current_chunk = candidate
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = para
        if current_chunk:
            chunks.append(current_chunk.strip())
        return chunks
    
    for file in os.listdir(pdf_folder):
        if not file.endswith(".pdf") and not file.endswith(".PDF"):
            continue
        file_path = os.path.join(pdf_folder, file)
        doc = pdf_open(file_path)
        full_text = "".join(page.get_text() for page in doc)
        doc.close()
        
        sections = split_by_sections(full_text)
        for section in sections:
            section_match = re.match(r"(\d{1,4})\.", section)
            section_number = section_match.group(1) if section_match else "UNKNOWN"
            chunks = semantic_chunk_section(section, max_tokens=800)
            for chunk in chunks:
                documents.append(chunk)
                metadatas.append({"file": file, "section": section_number, "text": chunk})
                ids.append(str(uuid.uuid4()))
    
    print(f" Semantic chunking: {len(documents)} chunks from {pdf_folder}")
    return documents, metadatas, ids

### 3b. Fixed-Size Chunking

In [ ]:
def fixed_size_chunk_pdfs(pdf_folder, chunk_size=3000):
    """
    FIXED-SIZE CHUNKING — simple character-based splitting.
    
    Strategy:
    1. Extract full text from each PDF
    2. Split into fixed-size character windows (no overlap)
    3. Each chunk is exactly `chunk_size` characters (except the last)
    
    Limitations:
    - May break mid-sentence or mid-word
    - No awareness of document structure
    - No context overlap between chunks
    
    Returns: (documents, metadatas, ids)
    """
    documents, metadatas, ids = [], [], []
    
    for file in os.listdir(pdf_folder):
        if not file.endswith(".pdf") and not file.endswith(".PDF"):
            continue
        file_path = os.path.join(pdf_folder, file)
        doc = pdf_open(file_path)
        full_text = "".join(page.get_text() for page in doc)
        doc.close()
        
        # Simple fixed-size splitting with no overlap
        for i in range(0, len(full_text), chunk_size):
            chunk = full_text[i:i + chunk_size].strip()
            if chunk:
                documents.append(chunk)
                metadatas.append({"file": file, "chunk_index": i // chunk_size, "text": chunk})
                ids.append(str(uuid.uuid4()))
    
    print(f"Fixed-size chunking ({chunk_size} chars): {len(documents)} chunks")
    return documents, metadatas, ids

### 3c. Recursive Character Text Splitting (Better than Fixed-Size)

In [ ]:
def recursive_chunk_pdfs(pdf_folder, chunk_size=2000, overlap=200):
    """
    RECURSIVE CHARACTER TEXT SPLITTING — structure-aware with overlap.
    
    Strategy:
    1. Extract full text from each PDF
    2. Try splitting at natural boundaries in order: \n\n → \n → '. ' → ' '
    3. Merge small splits into chunks up to chunk_size
    4. Add overlap between consecutive chunks for context continuity
    
    Advantages over fixed-size:
    - Preserves sentence and paragraph integrity
    - Overlap ensures context is not lost at boundaries
    - More semantically coherent chunks
    
    Returns: (documents, metadatas, ids)
    """
    documents, metadatas, ids = [], [], []
    separators = ["\n\n", "\n", ". ", " "]
    
    def recursive_split(text, separators, chunk_size):
        """Recursively split text using a hierarchy of separators."""
        final_chunks = []
        separator = separators[-1]  # default to last separator
        
        # Find the best separator that exists in the text
        for sep in separators:
            if sep in text:
                separator = sep
                break
        
        splits = text.split(separator)
        current_chunk = ""
        
        for split in splits:
            candidate = current_chunk + separator + split if current_chunk else split
            if len(candidate) <= chunk_size:
                current_chunk = candidate
            else:
                if current_chunk:
                    final_chunks.append(current_chunk.strip())
                # If a single split is still too large and we have more separators
                if len(split) > chunk_size and separator != separators[-1]:
                    remaining_seps = separators[separators.index(separator) + 1:]
                    sub_chunks = recursive_split(split, remaining_seps, chunk_size)
                    final_chunks.extend(sub_chunks)
                    current_chunk = ""
                else:
                    current_chunk = split
        
        if current_chunk:
            final_chunks.append(current_chunk.strip())
        
        return final_chunks
    
    def add_overlap(chunks, overlap_size):
        """Add overlap between consecutive chunks for context continuity."""
        if not chunks or overlap_size <= 0:
            return chunks
        overlapped = [chunks[0]]
        for i in range(1, len(chunks)):
            prev_suffix = chunks[i - 1][-overlap_size:]
            overlapped.append(prev_suffix + " " + chunks[i])
        return overlapped
    
    for file in os.listdir(pdf_folder):
        if not file.endswith(".pdf") and not file.endswith(".PDF"):
            continue
        file_path = os.path.join(pdf_folder, file)
        doc = pdf_open(file_path)
        full_text = "".join(page.get_text() for page in doc)
        doc.close()
        
        raw_chunks = recursive_split(full_text, separators, chunk_size)
        chunks_with_overlap = add_overlap(raw_chunks, overlap)
        
        for idx, chunk in enumerate(chunks_with_overlap):
            if chunk.strip():
                documents.append(chunk)
                metadatas.append({"file": file, "chunk_index": idx, "text": chunk})
                ids.append(str(uuid.uuid4()))
    
    print(f"Recursive chunking ({chunk_size} chars, {overlap} overlap): {len(documents)} chunks")
    return documents, metadatas, ids

## 4. Qdrant Ingestion

Create separate collections for each strategy and upload embedded chunks. Each collection is independent so we can query them separately for comparison.

In [ ]:
def create_collection(collection_name):
    """Create (or recreate) a Qdrant collection for the given strategy."""
    if qdrant_client.collection_exists(collection_name):
        qdrant_client.get_collection(collection_name)
        print(f"Collection '{collection_name}' already there ({EMBED_DIM}-dim, cosine)")
    else:
        qdrant_client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE)
        )
        print(f" Collection '{collection_name}' created ({EMBED_DIM}-dim, cosine)")

def upload_to_qdrant(documents, metadatas, ids, collection_name, embed_batch=100):
    """Embed and upload document chunks to a Qdrant collection."""
    total = len(documents)
    for i in range(0, total, embed_batch):
        batch_docs = documents[i:i + embed_batch]
        batch_meta = metadatas[i:i + embed_batch]
        batch_ids = ids[i:i + embed_batch]
        
        embeddings = embed_texts(batch_docs)
        points = [
            PointStruct(id=bid, vector=emb, payload=meta)
            for bid, emb, meta in zip(batch_ids, embeddings, batch_meta)
        ]
        qdrant_client.upsert(collection_name=collection_name, points=points)
        print(f"\nUploaded {min(i + embed_batch, total)}/{total}", end="\r")
    
    print(f"\n{total} chunks uploaded to '{collection_name}'")

### Ingestion for All 3 Strategies


In [ ]:
# ── Ingest for each chunking strategy ──
chunking_functions = {
    "semantic":  lambda: semantic_chunk_pdfs(PDF_FOLDER),
    "fixed":     lambda: fixed_size_chunk_pdfs(PDF_FOLDER, chunk_size=3000),
    "recursive": lambda: recursive_chunk_pdfs(PDF_FOLDER, chunk_size=2000, overlap=200),
}

chunk_stats = {}

for strategy_name, chunk_fn in chunking_functions.items():
    collection = COLLECTIONS[strategy_name]
    print(f"\n{'='*60}")
    print(f" Processing: {strategy_name.upper()} chunking → {collection}")
    print(f"{'='*60}")
    
    # 1. Chunk the PDFs
    docs, metas, doc_ids = chunk_fn()
    chunk_stats[strategy_name] = len(docs)
    
    # 2. Create collection
    create_collection(collection)
    
    # 3. Upload to Qdrant
    upload_to_qdrant(docs, metas, doc_ids, collection)

print(f"\n{'='*60}")
print("📊 Chunk Statistics:")
for name, count in chunk_stats.items():
    print(f"  {name:12s}: {count:,} chunks")